# 04 — Analyse Quantitative & Tests Statistiques

Ce notebook reproduit la méthodologie rigoureuse de l'article IBM (2023) :

1. **20 tirages aléatoires** → distribution empirique des performances
2. **Bootstrap** → intervalles de confiance à 95%
3. **Tests statistiques** → t-test apparié + Wilcoxon
4. **Taille d'effet** → Cohen's d
5. **Tables Victoire/Égalité/Défaite** → comme les Tables II-IV du papier IBM
6. **Stabilité des poids** → les kernels sélectionnés sont-ils robustes ?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from src.preprocessing import QuantumScaler, FeatureReducer
from src.kernels import build_feature_map, build_quantum_kernel, compute_kernel_matrix
from src.kernels.feature_maps import get_feature_map_library
from src.mkl.alignment import centered_alignment, projection_alignment, sdp_alignment
from src.evaluation.statistical_analysis import (
    multi_run_evaluation, summarize_multi_run,
    win_tie_loss_table, bootstrap_ci_all,
    pairwise_ttest, pairwise_wilcoxon, cohens_d,
    weight_stability_analysis,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42

## 1. Préparation du dataset et calcul des kernels

> On calcule **une seule fois** toutes les matrices de kernel sur le dataset complet,
> puis on fait les splits aléatoires par indexation. C'est efficace et reproductible.

In [ ]:
# ── Données synthétiques ──
N_SAMPLES = 200   # ↓ pour rapidité en simulation (IBM: 400)
N_QUBITS  = 4    # ↓ pour rapidité (IBM: 6-20)

X_raw, y = make_classification(
    n_samples=N_SAMPLES, n_features=20, n_informative=12,
    n_redundant=5, random_state=SEED, weights=[0.7, 0.3]
)

# ── Preprocessing ──
reducer = FeatureReducer(n_components=N_QUBITS)
scaler  = QuantumScaler(feature_range=(0, 2))
X = scaler.fit_transform(reducer.fit_transform(X_raw))

print(f'Dataset: {X.shape}, Classes: {np.bincount(y)}')

# ── Calcul des kernels sur tout le dataset ──
fm_library = get_feature_map_library(N_QUBITS)
print(f'Calcul de {len(fm_library)} kernels...')

K_matrices_full = []
kernel_names    = []

for label, fm in fm_library:
    qk = build_quantum_kernel(fm, kernel_type='fidelity')
    K  = compute_kernel_matrix(qk, X)
    K_matrices_full.append(K)
    kernel_names.append(label)
    print(f'  ✓ {label}')

print(f'\n{len(K_matrices_full)} matrices ({K_matrices_full[0].shape}) prêtes.')

## 2. Multi-run : 20 tirages aléatoires

On définit les méthodes à comparer, chacune est une fonction `(K_list_train, y_train) -> weights`.

In [ ]:
def _build_target(y):
    return np.outer(y, y).astype(float)

# Définition des méthodes
methods = {
    'Average':    lambda Ks, y: np.ones(len(Ks)) / len(Ks),
    'Centered':   lambda Ks, y: centered_alignment(Ks, _build_target(y)),
    'Projection': lambda Ks, y: projection_alignment(Ks, _build_target(y)),
    'SDP':        lambda Ks, y: sdp_alignment(Ks, _build_target(y)),
    'Single-Best':lambda Ks, y: _single_best(Ks, y),  # baseline single kernel
}

def _single_best(Ks, y):
    """Sélectionne le kernel avec le meilleur alignement individuel."""
    Ky = _build_target(y)
    scores = [np.sum(K * Ky) / (np.linalg.norm(K, 'fro') * np.linalg.norm(Ky, 'fro') + 1e-10)
              for K in Ks]
    w = np.zeros(len(Ks))
    w[np.argmax(scores)] = 1.0
    return w

N_RUNS = 20  # Comme dans l'article IBM
print(f'Running {N_RUNS} random splits for {len(methods)} methods...')

results = multi_run_evaluation(
    kernel_matrices_full=K_matrices_full,
    y_full=y,
    methods=methods,
    n_runs=N_RUNS,
    test_size=0.33,
    C=1.0,
    scoring='roc_auc',
)
print('\nDone!')

## 3. Résumé statistique : Moyenne ± Écart-type

In [ ]:
summary = summarize_multi_run(results)

# Tableau de résultats
df_summary = pd.DataFrame([
    {
        'Method': name,
        'Mean AUC': f"{v['mean']:.4f}",
        'Std':      f"± {v['std']:.4f}",
        'Min':      f"{v['min']:.4f}",
        'Max':      f"{v['max']:.4f}",
        'Median':   f"{v['median']:.4f}",
    }
    for name, v in summary.items()
]).sort_values('Mean AUC', ascending=False)

print('\n=== ROC-AUC sur 20 tirages ===\n')
print(df_summary.to_string(index=False))

## 4. Bootstrap : intervalles de confiance à 95%

In [ ]:
ci_results = bootstrap_ci_all(results, n_bootstrap=2000, ci=0.95)

# Barplot avec intervalles de confiance
names  = list(ci_results.keys())
means  = [ci_results[n]['mean'] for n in names]
ci_low = [ci_results[n]['mean'] - ci_results[n]['ci_low']  for n in names]
ci_hi  = [ci_results[n]['ci_high'] - ci_results[n]['mean'] for n in names]

order = np.argsort(means)[::-1]
names_ord = [names[i] for i in order]
means_ord = [means[i] for i in order]
err_low   = [[ci_low[i] for i in order]]
err_hi    = [[ci_hi[i]  for i in order]]

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(names_ord)))
bars = ax.bar(names_ord, means_ord, color=colors[::-1], edgecolor='black', linewidth=0.7)
ax.errorbar(names_ord, means_ord, yerr=[err_low[0], err_hi[0]],
            fmt='none', color='black', capsize=6, linewidth=2)

ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title(f'Comparaison des méthodes QMKL\n(Bootstrap CI 95%, {N_RUNS} tirages)', fontsize=13)
ax.set_ylim(max(0, min(means_ord) - 0.1), min(1.0, max(means_ord) + 0.1))
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Aléatoire')
ax.legend()
plt.tight_layout()
plt.savefig('../results/04_bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Distribution des scores (Violinplot)

In [ ]:
# Convertir en DataFrame long pour seaborn
rows = []
for name, scores in results.items():
    for s in scores:
        rows.append({'Method': name, 'ROC-AUC': s})
df_long = pd.DataFrame(rows)

order_methods = sorted(results.keys(),
                       key=lambda n: np.mean(results[n]), reverse=True)

fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=df_long, x='Method', y='ROC-AUC', order=order_methods,
               palette='Set2', inner='box', ax=ax)
sns.stripplot(data=df_long, x='Method', y='ROC-AUC', order=order_methods,
              color='black', alpha=0.4, size=4, ax=ax)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5)
ax.set_title(f'Distribution des ROC-AUC ({N_RUNS} tirages)', fontsize=13)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../results/04_violinplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tests statistiques : t-test apparié + Wilcoxon

**H0** : les deux méthodes ont la même performance.
**H1** : la méthode A est meilleure que B.

Seuil classique : p < 0.05 → différence statistiquement significative.

In [ ]:
pv_ttest    = pairwise_ttest(results, alternative='greater')
pv_wilcoxon = pairwise_wilcoxon(results, alternative='greater')

names = list(results.keys())

# Matrice de p-values (t-test)
pmat = np.array([[pv_ttest[a][b] for b in names] for a in names])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pmat_data, title in zip(
    axes,
    [pmat, np.array([[pv_wilcoxon[a][b] for b in names] for a in names])],
    ['t-test apparié (p-value : ligne > colonne)', 'Wilcoxon (p-value : ligne > colonne)']
):
    sns.heatmap(pmat_data, xticklabels=names, yticklabels=names,
                annot=True, fmt='.3f', cmap='RdYlGn_r',
                vmin=0, vmax=0.1, ax=ax,
                linewidths=0.5, cbar_kws={'label': 'p-value'})
    ax.set_title(title, fontsize=11)

plt.suptitle('Tests statistiques (vert = significatif, p < 0.05)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/04_statistical_tests.png', dpi=150, bbox_inches='tight')
plt.show()

# Interprétation textuelle
print('\n=== Interprétation (p < 0.05 = significatif) ===')
for a in names:
    for b in names:
        if a != b and pv_ttest[a][b] < 0.05:
            d = cohens_d(results[a], results[b])
            size = 'grand' if abs(d)>0.8 else ('moyen' if abs(d)>0.5 else 'petit')
            print(f'  {a} > {b}: p={pv_ttest[a][b]:.4f}, Cohen\'d={d:.3f} ({size} effet)')

## 7. Table Victoire / Égalité / Défaite

Reproduit les Tables II-IV de l'article IBM.
Pour chaque run, on compte combien de fois la méthode bat le baseline `Single-Best`.

In [ ]:
wtl = win_tie_loss_table(results, baseline_method='Single-Best', threshold=0.001)

df_wtl = pd.DataFrame([
    {'Method': name, 'Wins': v['wins'], 'Ties': v['ties'],
     'Losses': v['losses'], 'Win Rate': f"{v['win_rate']:.0%}"}
    for name, v in wtl.items()
]).sort_values('Wins', ascending=False)

print(f'\n=== Victoires vs Single-Best ({N_RUNS} tirages) ===\n')
print(df_wtl.to_string(index=False))

# Stacked bar
fig, ax = plt.subplots(figsize=(9, 4))
mnames = df_wtl['Method'].values
wins   = df_wtl['Wins'].values
ties   = df_wtl['Ties'].values
losses = df_wtl['Losses'].values

x = np.arange(len(mnames))
ax.bar(x, wins,         label='Wins',   color='#2ecc71')
ax.bar(x, ties,         label='Ties',   color='#95a5a6', bottom=wins)
ax.bar(x, losses,       label='Losses', color='#e74c3c', bottom=wins+ties)
ax.set_xticks(x)
ax.set_xticklabels(mnames)
ax.set_ylabel('Nombre de runs')
ax.set_title('Victoires vs Single-Best Kernel')
ax.legend()
ax.axhline(N_RUNS * 0.5, color='black', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../results/04_win_tie_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Stabilité des poids des kernels

Les mêmes kernels sont-ils sélectionnés à chaque split ?
→ Si oui : sélection robuste. Si non : le modèle est instable.

In [ ]:
from src.mkl.alignment import centered_alignment

weight_fn = lambda Ks, y: centered_alignment(Ks, np.outer(y, y).astype(float))

weights_matrix, w_summary = weight_stability_analysis(
    kernel_matrices_full=K_matrices_full,
    y_full=y,
    weight_fn=weight_fn,
    kernel_names=kernel_names,
    n_runs=N_RUNS,
)

# Heatmap des poids sur les runs
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap run × kernel
ax = axes[0]
sns.heatmap(weights_matrix, xticklabels=kernel_names, yticklabels=[f'Run {i}' for i in range(N_RUNS)],
            cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Poids w_i'})
ax.set_title('Poids des kernels par run (Centered Alignment)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Barplot moyenne ± std
ax2 = axes[1]
mean_w = weights_matrix.mean(axis=0)
std_w  = weights_matrix.std(axis=0, ddof=1)
order  = np.argsort(mean_w)[::-1]

ax2.bar(range(len(kernel_names)), mean_w[order],
        yerr=std_w[order], capsize=4,
        color=plt.cm.YlOrRd(mean_w[order] / (mean_w.max() + 1e-10)),
        edgecolor='black', linewidth=0.5)
ax2.set_xticks(range(len(kernel_names)))
ax2.set_xticklabels([kernel_names[i] for i in order], rotation=45, ha='right')
ax2.set_ylabel('Poids moyen')
ax2.set_title('Importance moyenne des kernels (↑ = plus important)')

plt.suptitle('Analyse de stabilité des poids kernels', fontsize=13)
plt.tight_layout()
plt.savefig('../results/04_weight_stability.png', dpi=150, bbox_inches='tight')
plt.show()

# Top kernels
print('\n=== Top 5 kernels (par poids moyen) ===')
top5 = sorted(w_summary.items(), key=lambda x: x[1]['mean'], reverse=True)[:5]
for name, v in top5:
    print(f"  {name:25s}: {v['mean']:.4f} ± {v['std']:.4f}  (non-zero: {v['nonzero_rate']:.0%})")